# Carve3D full reconstruction + MRC pipeline

This notebook runs a complete, open-checkpoint replacement for the components not released by Carve3D: four real views → LGM Gaussian reconstruction → same-camera renders → LPIPS MRC.

Before running, enable **Internet** and a **GPU accelerator** in Kaggle. Put four images in a Kaggle Dataset and name them `view_000`, `view_090`, `view_180`, and `view_270` (png/jpg/jpeg/webp). They must show the same centered object at azimuths 0°, 90°, 180°, 270°, with roughly the same elevation.

In [ ]:
# Clone this branch after it has been pushed.
REPO_URL = 'https://github.com/huytrao/carve3d-test.git'
REPO_DIR = '/kaggle/working/carve3d-test'
LGM_DIR = '/kaggle/working/LGM'
!git clone --branch implement_full_pipeline $REPO_URL $REPO_DIR
%cd $REPO_DIR
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!bash scripts/setup_lgm_kaggle.sh $LGM_DIR

## Prompt quick start (T4 x2)

Use the following **Code** cell directly. Do not paste the contents of `run_full_pipeline.py` into a notebook cell; running the saved file avoids Jupyter import/path issues. Enable Internet and choose the **T4 x2** accelerator.

In [ ]:
# One-cell prompt run. Change only CARVE3D_PROMPT if desired.
# The prompt notebook force-reinstalls LGM-compatible kiui==0.2.3 before import.
# Restart Session if a failed run already imported old CUDA/Python modules.
%env CARVE3D_PROMPT=a wooden chair, studio product photograph, centered object, white background
%env CARVE3D_OUTPUT_DIR=/kaggle/working/chair-output
!wget -q -O /kaggle/working/run_full_pipeline.py https://raw.githubusercontent.com/huytrao/carve3d-test/implement_full_pipeline/run_full_pipeline.py
!python /kaggle/working/run_full_pipeline.py

In [ ]:
# Change only this path to your Kaggle Dataset folder.
INPUT_DIR = '/kaggle/input/YOUR-FOUR-VIEWS-DATASET'
OUTPUT_DIR = '/kaggle/working/carve3d-output'
!find $INPUT_DIR -maxdepth 1 -type f | sort
!python full_pipeline/run.py --lgm-root $LGM_DIR --input-dir $INPUT_DIR --output-dir $OUTPUT_DIR

In [ ]:
# Inspect the numerical MRC result (lower is better) and visual diagnostics.
# This works for either the prompt run or the four-real-view run.
import json
import os
from pathlib import Path
from IPython.display import Image, Video, display

candidate_dirs = [
    os.environ.get('CARVE3D_OUTPUT_DIR'),
    globals().get('PROMPT_OUTPUT'),
    globals().get('OUTPUT_DIR'),
]
result_dir = next((Path(path) for path in candidate_dirs if path and (Path(path) / 'metrics.json').is_file()), None)
if result_dir is None:
    raise FileNotFoundError(
        'No completed pipeline output found. Run the prompt/four-view pipeline until it prints Completed, then rerun this cell.'
    )

with open(result_dir / 'metrics.json') as f:
    metrics = json.load(f)
print('Results:', result_dir)
print(json.dumps(metrics['mrc'], indent=2))
display(Image(filename=result_dir / 'input_grid.png'))
display(Image(filename=result_dir / 'render_grid.png'))
display(Video(result_dir / 'orbit.mp4', embed=True))
print('PLY 3D asset:', result_dir / 'reconstruction.ply')

## Optional prompt route

This route uses MVDream's public checkpoint to generate four input views, then follows the same LGM/MRC pipeline. Select Kaggle **T4 x2**: GPU 0 samples MVDream and GPU 1 reconstructs/renders with LGM. It is a runnable baseline, not the unpublished Carve3D RL-finetuned checkpoint.

In [ ]:
%env CARVE3D_PROMPT=a wooden chair, studio product photograph, centered object, white background
%env CARVE3D_OUTPUT_DIR=/kaggle/working/carve3d-prompt-output
!python run_full_pipeline.py